In [ ]:
import os
import random
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
from tensorflow.keras import layers, models
train_dir = '/kaggle/input/competitions/cs-460-muffin-vs-chihuahua-classification-challenge/train'
test_dir = '/kaggle/input/competitions/cs-460-muffin-vs-chihuahua-classification-challenge/kaggle_test_final'

In [ ]:
muffin_count = len(os.listdir(os.path.join(train_dir, "muffin")))
chihuahua_count = len(os.listdir(os.path.join(train_dir, "chihuahua")))

total = muffin_count + chihuahua_count

print("Muffin images:", muffin_count)
print("Chihuahua images:", chihuahua_count)
print("Total images:", total)

print("\nClass distribution:")
print("Muffin:", muffin_count/total*100, "%")
print("Chihuahua:", chihuahua_count/total*100, "%")
train_datagen = ImageDataGenerator(
    
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    shear_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8,1.2],
    validation_split=0.2
)

In [ ]:
validation_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

In [ ]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(256,256),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

In [ ]:
validation_generator = validation_datagen.flow_from_directory(
    train_dir,
    target_size=(256,256),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

In [ ]:
model = models.Sequential([

    layers.Conv2D(32, (3,3), padding='same', input_shape=(256,256,3)),
    layers.Activation('relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), padding='same'),
    layers.Activation('relu'),
    layers.MaxPooling2D((2,2)),
    
    layers.Conv2D(128, (3,3), padding='same'),
    layers.Activation('relu'),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),

    layers.Dense(128),
    layers.Activation('relu'),
    layers.Dropout(0.5),

    layers.Dense(1, activation='sigmoid')
])

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.summary()

In [ ]:
%%time
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=5
)

In [ ]:
# Accuracy
train_acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

# Loss
train_loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(1, len(train_acc) + 1)

plt.figure(figsize=(12,5))

# Accuracy plot
plt.subplot(1,2,1)
plt.plot(epochs, train_acc, marker='o', label='Training Accuracy')
plt.plot(epochs, val_acc, marker='o', label='Validation Accuracy')
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

# Loss plot
plt.subplot(1,2,2)
plt.plot(epochs, train_loss, marker='o', label='Training Loss')
plt.plot(epochs, val_loss, marker='o', label='Validation Loss')
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
v_loss, v_acc = model.evaluate(validation_generator)
print("Accuracy:", v_acc)

In [ ]:
from tensorflow.keras.applications import ResNet50, MobileNetV2

In [ ]:
base_resnet = ResNet50(weights='imagenet', include_top=False, input_shape=(256, 256, 3))
base_resnet.trainable = False  # freeze pretrained layers

resnet_model = models.Sequential([
    base_resnet,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
], name='ResNet50')

In [ ]:
resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

resnet_model.summary()

In [ ]:
history_resnet = resnet_model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=10
)

In [ ]:
v_loss, v_acc = resnet_model.evaluate(validation_generator)
print("Accuracy:", v_acc)

In [ ]:
# Accuracy
train_acc = history_resnet.history['accuracy']
val_acc = history_resnet.history['val_accuracy']

# Loss
train_loss = history_resnet.history['loss']
val_loss = history_resnet.history['val_loss']

epochs = range(1, len(train_acc) + 1)

plt.figure(figsize=(12,5))

# Accuracy plot
plt.subplot(1,2,1)
plt.plot(epochs, train_acc, marker='o', label='Training Accuracy')
plt.plot(epochs, val_acc, marker='o', label='Validation Accuracy')
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

# Loss plot
plt.subplot(1,2,2)
plt.plot(epochs, train_loss, marker='o', label='Training Loss')
plt.plot(epochs, val_loss, marker='o', label='Validation Loss')
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
base_mobilenet = MobileNetV2(weights='imagenet', include_top=False, input_shape=(256, 256, 3))
base_mobilenet.trainable = False

mobilenet_model = models.Sequential([
    base_mobilenet,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
], name='MobileNetV2')

In [ ]:
mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

mobilenet_model.summary()

In [ ]:
history_mobilenet = mobilenet_model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=5
)

In [ ]:
v_loss, v_acc = mobilenet_model.evaluate(validation_generator)
print("Accuracy:", v_acc)

In [ ]:
# Accuracy
train_acc = history_mobilenet.history['accuracy']
val_acc = history_mobilenet.history['val_accuracy']

# Loss
train_loss = history_mobilenet.history['loss']
val_loss = history_mobilenet.history['val_loss']

epochs = range(1, len(train_acc) + 1)

plt.figure(figsize=(12,5))

# Accuracy plot
plt.subplot(1,2,1)
plt.plot(epochs, train_acc, marker='o', label='Training Accuracy')
plt.plot(epochs, val_acc, marker='o', label='Validation Accuracy')
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

# Loss plot
plt.subplot(1,2,2)
plt.plot(epochs, train_loss, marker='o', label='Training Loss')
plt.plot(epochs, val_loss, marker='o', label='Validation Loss')
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
from tensorflow.keras.applications import EfficientNetV2S
base = EfficientNetV2S(weights='imagenet', include_top=False, input_shape=(256,256, 3))
base.trainable = False

efficientnet_model = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    #layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
], name= "EfficientNetV2S")

In [ ]:
efficientnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

efficientnet_model.summary()

In [ ]:
history_efficientnet = efficientnet_model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=10
)

In [ ]:
v_loss, v_acc = efficientnet_model.evaluate(validation_generator)
print("Accuracy:", v_acc)

In [ ]:
# Accuracy
train_acc = history_efficientnet.history['accuracy']
val_acc = history_efficientnet.history['val_accuracy']

# Loss
train_loss = history_efficientnet.history['loss']
val_loss = history_efficientnet.history['val_loss']

epochs = range(1, len(train_acc) + 1)

plt.figure(figsize=(12,5))

# Accuracy plot
plt.subplot(1,2,1)
plt.plot(epochs, train_acc, marker='o', label='Training Accuracy')
plt.plot(epochs, val_acc, marker='o', label='Validation Accuracy')
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

# Loss plot
plt.subplot(1,2,2)
plt.plot(epochs, train_loss, marker='o', label='Training Loss')
plt.plot(epochs, val_loss, marker='o', label='Validation Loss')
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:

import os
# Get all image filenames directly from test_dir
test_images = os.listdir(test_dir)

test_df = pd.DataFrame({'filename': test_images})

test_gen = ImageDataGenerator(rescale=1./255).flow_from_dataframe(
    test_df,
    directory=test_dir,
    x_col='filename',
    y_col=None,
    target_size=(256,256),
    batch_size=32,
    class_mode=None,
    shuffle=False
)

preds = (selected_model.predict(test_gen) > 0.5).astype(int).flatten()
idx_to_class = {v: k for k, v in train_generator.class_indices.items()}

submission = pd.DataFrame({
    'ID':    test_df['filename'],
    'Label': [idx_to_class[p] for p in preds]
})

print(submission.head(10))
submission.to_csv('submission.csv', index=False)
print("submission.csv saved!")

In [ ]:
import random
from tensorflow.keras.preprocessing import image

# Pick a random image from test_dir
random_img = random.choice(test_images)
img_path = os.path.join(test_dir, random_img)

# Load and preprocess
img = image.load_img(img_path, target_size=(256,256))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = selected_model.predict(img_array)[0][0]
label = idx_to_class[int(pred > 0.5)]
confidence = pred if pred > 0.5 else 1 - pred

# Show
plt.imshow(img)
plt.title(f"Prediction: {label} ({confidence*100:.2f}%)")
plt.axis('off')
plt.show()